# Using HTMD to setup a protein-ligand simulation in explicit solvent

HTMD: High-Throughput Molecular Dynamics for Molecular Discovery
S. Doerr, M. J. Harvey, Frank Noé, and G. De Fabritiis
Journal of Chemical Theory and Computation 2016 12 (4), 1845-1852
DOI: 10.1021/acs.jctc.6b00049.

    
https://software.acellera.com/docs/latest/htmd/index.html

Before you start, make sure that HTMD is installed on your machine. 
HTMD can be installed using conda.
Best to create a conda HTMD environment

conda create -n htmd python=3.10
mamba install htmd -c acellera -c conda-forge htmd
conda activate htmd




In [2]:
from htmd.ui import *

In [3]:
!ls

atp.lib       KPC-D179A_AB.ipynb	  ted.frcmod  ted.pdb
ATP_mod.prep  leap.log			  ted.inpcrd  ted.prmtop
frcmod.phos   Setup_protein-ligand.ipynb  ted.lib
input.pdb     ted_ac.pdb		  ted.mol2


## Import your PDB structure

In [4]:
mol = Molecule('input.pdb')
mol.view(gui=True)

In [5]:
print (mol)

Molecule with 17884 atoms and 1 frames
Atom field - altloc shape: (17884,)
Atom field - atomtype shape: (17884,)
Atom field - beta shape: (17884,)
Atom field - chain shape: (17884,)
Atom field - charge shape: (17884,)
Atom field - coords shape: (17884, 3, 1)
Atom field - element shape: (17884,)
Atom field - formalcharge shape: (17884,)
Atom field - insertion shape: (17884,)
Atom field - masses shape: (17884,)
Atom field - name shape: (17884,)
Atom field - occupancy shape: (17884,)
Atom field - record shape: (17884,)
Atom field - resid shape: (17884,)
Atom field - resname shape: (17884,)
Atom field - segid shape: (17884,)
Atom field - serial shape: (17884,)
angles shape: (0, 3)
bonds shape: (0, 2)
bondtype shape: (0,)
box shape: (3, 1)
boxangles shape: (3, 1)
crystalinfo: {}
dihedrals shape: (0, 4)
fileloc shape: (1, 2)
impropers shape: (0, 4)
reps: 
ssbonds shape: (0,)
step shape: (1,)
time shape: (1,)
topoloc: /home/alessandro/Desktop/dockedMD/new/ted_docked/parametrisation/param/0_pa

In [6]:
mol.get('resname')

array(['VAL', 'VAL', 'VAL', ..., 'atp', 'atp', 'MG'], dtype=object)

In [7]:
np.unique(mol.get('resname', sel='resid 1'))

array([], dtype=object)

In [8]:
mol.write('start.pdb')

In [9]:
len(mol.get("serial",sel="protein"))

17772

## Recentering the molecule

In [10]:
mol.center()
mol.view(viewer='VMD')

In [11]:
from moleculekit.util import maxDistance
D = maxDistance(mol, 'all', origin=[0, 0, 0])
print(D)

82.132645


HTMD requires different segments for chemical entities

In [12]:
mol.set('segid', 'P', sel='protein')
mol.set('segid', 'A', sel='resname atp')
mol.set('segid', 'M', sel='resname MG')
mol.set('segid', 'W', sel='water')

mol.set('segid','L', sel='resname RES')

## Setting up the box size and solvation

In [13]:
smol = solvate(mol, pad=10) # box size 12 Ang from edge of solute
smol.reps.add(sel='water', style='Lines')
smol.reps.add(sel='resname RES atp MG or protein', style='Licorice')
smol.view(viewer='VMD')

2024-11-06 12:33:01,142 - htmd.builder.solvate - INFO - Using water pdb file at: /home/alessandro/miniconda3/envs/htmd/lib/python3.10/site-packages/htmd/share/solvate/wat.pdb
2024-11-06 12:33:01,768 - htmd.builder.solvate - INFO - Replicating 12 water segments, 2 by 2 by 3
Solvating: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 12/12 [00:19<00:00,  1.63s/it]
2024-11-06 12:33:24,728 - htmd.builder.solvate - INFO - 54088 water molecules were added to the system.
2024-11-06 12:33:27,704 - moleculekit.writers - WARNING - Field "serial" of PDB overflows. Your data will be truncated to 5 characters.
X_ChangeProperty: BadValue (integer parameter out of range for operation) 0x40
X_ChangeProperty: BadValue (integer parameter out of range for operation) 0x40
X_ChangeProperty: BadValue (integer parameter out of range for operation) 0x40


## Check and add appropriate force field

In [14]:
amber.listFiles()

---- Forcefield files list: /usr/local/amber20/dat/leap/cmd/ ----
leaprc.amberdyes
leaprc.conste
leaprc.constph
leaprc.DNA.bsc1
leaprc.DNA.OL15
leaprc.ffAM1
leaprc.ffPM3
leaprc.gaff
leaprc.gaff2
leaprc.GLYCAM_06EPb
leaprc.GLYCAM_06j-1
leaprc.lipid14
leaprc.lipid17
leaprc.mimetic.ff15ipq
leaprc.modrna08
leaprc.music
leaprc.phosaa10
leaprc.phosaa14SB
leaprc.phosaa19SB
leaprc.protein.fb15
leaprc.protein.ff03.r1
leaprc.protein.ff03ua
leaprc.protein.ff14SB
leaprc.protein.ff14SB_modAA
leaprc.protein.ff14SBonlysc
leaprc.protein.ff15ipq
leaprc.protein.ff15ipq-vac
leaprc.protein.ff19ipq
leaprc.protein.ff19SB
leaprc.protein.ff19SB_modAA
leaprc.RNA.LJbb
leaprc.RNA.OL3
leaprc.RNA.ROC
leaprc.RNA.Shaw
leaprc.RNA.YIL
leaprc.water.fb3
leaprc.water.fb4
leaprc.water.opc
leaprc.water.opc3
leaprc.water.spce
leaprc.water.spceb
leaprc.water.tip3p
leaprc.water.tip4pd
leaprc.water.tip4pew
leaprc.xFPchromophores
---- OLD Forcefield files list: /usr/local/amber20/dat/leap/cmd/ ----
oldff/leaprc.DNA.bsc0
oldff/l

Add full path to force fields if they are installed in a non-conventional place

In [17]:
ffs = ['/home/alessandro/miniconda3/envs/htmd/dat/leap/cmd/leaprc.protein.ff19SB', '/home/alessandro/miniconda3/envs/htmd/dat/leap/cmd/leaprc.water.tip3p', '/home/alessandro/miniconda3/envs/htmd/dat/leap/cmd/leaprc.gaff']
frcmods_amber = ffs + ['./ted.frcmod', './frcmod.phos' ]
offlibs=['./ted.lib', './atp.lib']

## Build the system

In [25]:
molbuilt = amber.build(smol, param=frcmods_amber, ionize=True, saltcation='K', saltconc=0.15, outdir='./build', offlibraries=offlibs, caps={'P':['none','none']})

2024-11-06 13:04:29,548 - htmd.builder.amber - INFO - Detecting disulfide bonds.
2024-11-06 13:04:29,583 - htmd.builder.builder - INFO - 0 disulfide bonds were added
2024-11-06 13:04:30,204 - htmd.builder.amber - WARNING - Was not able to find ff file leaprc.lipid21
2024-11-06 13:04:32,567 - moleculekit.writers - WARNING - Field "serial" of PDB overflows. Your data will be truncated to 5 characters.
2024-11-06 13:04:35,418 - htmd.builder.amber - INFO - Starting the build.
2024-11-06 13:06:41,961 - htmd.builder.amber - INFO - Finished building.
2024-11-06 13:06:45,738 - moleculekit.writers - WARNING - Field "serial" of PDB overflows. Your data will be truncated to 5 characters.
2024-11-06 13:06:46,176 - moleculekit.writers - WARNING - Field "resid" of PDB overflows. Your data will be truncated to 4 characters.
2024-11-06 13:08:07,608 - htmd.builder.builder - WARNING - Found cis peptide bond in 1 frames: [0] in the omega diheral "Angle of (ILE 679 CA  ) (ILE 679 C  ) (PRO 680 N  ) (PRO 6

In [26]:
from htmd.protocols.equilibration_v3 import Equilibration
md = Equilibration()
md.acemd.show()

#
barostat             	on
barostatpressure     	1.01325
cutoff               	9
minimize             	500
pme                  	on
restart              	on
slowperiod           	1
switchdistance       	7.5
switching            	on
thermostat           	on
thermostatdamping    	1
timestep             	4
trajectoryfile       	output.xtc
trajectoryperiod     	25000



In [29]:
md = Equilibration()
md.runtime = 5
md.timeunits = 'ns'
md.temperature = 300
md.useconstantratio = False  # only for membrane sims


# # Add a 10A flat bottom potential to prevent the ligand from diffusing from original position during equilibration
width = np.array([10, 10, 30])
flatbot = GroupRestraint('resname RES and noh', width, [(5, '0ns')])
md.restraints = [flatbot] + md.defaultEquilRestraints('5ns')
md.write('./build/', './equil')

2024-11-06 13:46:12,016 - moleculekit.readers - WARNING - Non-integer values were read from the PDB "serial" field. Dropping PDB values and assigning new ones.
2024-11-06 13:46:12,083 - moleculekit.readers - WARNING - Reading PDB file with more than 99999 atoms. Bond information can be wrong.
2024-11-06 13:47:53,962 - htmd.builder.builder - WARNING - Found cis peptide bond in 1 frames: [0] in the omega diheral "Angle of (ILE 679 CA  ) (ILE 679 C  ) (PRO 680 N  ) (PRO 680 CA  ) " with indexes [10987, 11002, 11004, 11014]


In [30]:
# Load the structure
mol=Molecule('equil/structure.pdb')

# Visualize the flat bottom potential box
mol.view('not water and not ions')
fbcentre_bs1 = mol.get('coords', sel='resname RES and noh').mean(axis=0).squeeze()
b_bs1 = VMDBox(np.vstack((fbcentre_bs1 - width/2, fbcentre_bs1 + width/2)).T.flatten())

2024-11-06 13:48:07,506 - moleculekit.readers - WARNING - Non-integer values were read from the PDB "serial" field. Dropping PDB values and assigning new ones.
2024-11-06 13:48:07,577 - moleculekit.readers - WARNING - Reading PDB file with more than 99999 atoms. Bond information can be wrong.
2024-11-06 13:48:11,353 - moleculekit.writers - WARNING - Field "serial" of PDB overflows. Your data will be truncated to 5 characters.
X_ChangeProperty: BadValue (integer parameter out of range for operation) 0x40
X_ChangeProperty: BadValue (integer parameter out of range for operation) 0x40


## Equilibration

In [ ]:
mdx = LocalGPUQueue()
mdx.submit('./equil')
mdx.wait()

## For Simple Production Run

In [ ]:
md.write('./equil', './pdxn')

In [ ]:
mdx = AcemdLocal(devices=[1])
mdx.submit('./pdxn')
mdx.wait()

## For Adative Simulations
** first part for Adaptive Simulations
** Second for adaptive bandit

In [ ]:
from htmd.protocols.production_v6 import Production
from htmd.adaptive.adaptiverun import AdaptiveMD


ad = AdaptiveMD()
md = Production()
md.acemd.show()

In [ ]:
md.acemd.bincoordinates = 'output.coor'
md.acemd.extendedsystem  = 'output.xsc'
md.acemd.binvelocities=None
md.acemd.run = '50ns'
md.temperature = 300
md.acemd.trajectoryfile = 'trajectory.xtc'

## For Adaptive Simulations

In [ ]:
md.write('./equil', './generators/s1')

## This is Adaptive Sampling run code

queue = LocalGPUQueue()
queue.datadir = './data'
queue.devices = [0,1,2,3]

ad = AdaptiveMD()
ad.app = queue
ad.nmin=3  # minimum number of running sims
ad.nmax=4  # maximum number of running sims
ad.nepochs = 410 
ad.generatorspath = './generators/s1'
ad.ticadim = 3 # number of TICA dimensions
#ad.projection = MetricSelfDistance('(resid 219 220 221) and (backbone)')
#ad.projection = MetricSelfDistance('(residue % 3 == 1) and (name CA)')
mol = Molecule("generators/s1/structure.prmtop")
mol.read("generators/s1/structure.pdb")
mol.set("segid", "0", sel="protein")
mol.set("segid", "1", sel="all and not protein")
prodih=Dihedral.proteinDihedrals(mol,sel="backbone", dih=["phi","psi"])
ad.projection = [MetricDihedral(prodih)]

ad.updateperiod = 600 
ad.run()

# For Adaptive Bandit Simulations

In [ ]:
## Adaptive Bandit code

queue = LocalGPUQueue()
queue.datadir = './data'
queue.devices = [0,1,2,3]

ab = AdaptiveBandit()
ab.app = queue

ab.generatorspath = './generators/s1'
ab.ticadim = 3 # number of TICA dimensions

ab.nmin=3  # minimum number of running sims
ab.nmax=4  # maximum number of running sims
ab.nframes = 2500000000


ab.clustmethod = MiniBatchKMeans

mol = Molecule("generators/s1/structure.prmtop")
mol.read("generators/s1/structure.pdb")
mol.set("segid", "0", sel="protein")
mol.set("segid", "1", sel="all and not protein")

#prodih=Dihedral.proteinDihedrals(mol,sel="backbone", dih=["phi","psi"])
ab.projection = MetricSelfDistance('(residue % 4 == 1) and (name CA)')

ab.exploration = 0.01
ab.updateperiod = 600

In [ ]:
## AB code

ref = Molecule('./protein.pdb')

def contactGoal(mol, crystal):
    crystalCO = MetricSelfDistance('(residue % 4 == 1) and (name CA)', periodic=None,
                                   metric='contacts',
                                   threshold=10).project(crystal).squeeze()
    proj = MetricSelfDistance('(residue % 4 == 1) and (name CA)',
                              metric='contacts',
                              threshold=10).project(mol)
    # How many crystal contacts are seen?
    co_score = np.sum(proj[:, crystalCO] == 1, axis=1).astype(float)
    co_score /= np.sum(crystalCO)
    return co_score

ab.goalfunction = (contactGoal, (ref,))
ab.goal_init = 0.3

In [ ]:
## AB run
ab.run()

In [ ]:
crystal = Molecule('./protein.pdb')
mol = Molecule("input/e1s1_1/structure.prmtop")
mol.read("./data/e1s1_1/trajectory.xtc")
contactGoal(mol, crystal)

In [ ]:
simlist(glob('./filtered/*/'), glob('./filtered/*.pdb'))

In [ ]:
recon, chain, pathlist = reconstructAdaptiveTraj(simlist(glob('./filtered/*/'), glob('./filtered/*.pdb')),291)

In [ ]:
recon.write('penI-sim291.xtc')